In [ ]:
# @title Mise en place de l'environnement
%set_env UV_PROJECT_ENVIRONMENT /usr/local
%set_env MLFLOW_TRACKING_URI https://dagshub.com/m09/landscape-classifier.mlflow
%set_env MPLBACKEND notebook
!rm -rf sample_data .config
!pip install -U uv
!git config --global user.email "jeanne@durant.fr"
!git config --global user.name "Jeanne Durant"
!git config --global init.defaultBranch main
!git clone --branch solution https://github.com/shuuchuu/landscape-classifier.git
%cd /content/landscape-classifier
!uv sync --all-extras

# MLflow, registre de modèles & déploiement

Cette démonstration déroule la solution des travaux pratiques du même nom, sur la branche `solution` du [dépôt `landscape-classifier`](https://github.com/shuuchuu/landscape-classifier/tree/solution)&nbsp;: publication d'un modèle sur le registre de modèles MLflow, utilisation de ce registre, puis service du modèle par une API FastAPI empaquetée avec Docker.

## Identifiants pour le serveur MLflow

Entrez vos identifiants DagsHub (le mot de passe est le jeton secret, pas votre mot de passe DagsHub)&nbsp;:

In [ ]:
username = ""
token = ""

In [ ]:
# @title À exécuter
%set_env MLFLOW_TRACKING_USERNAME {username}
%set_env MLFLOW_TRACKING_PASSWORD {token}

## Récupération des données

In [ ]:
!wget -qO- https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh | bash -s landscape
!mv landscape/seg_train train-data

## Publication d'un modèle sur un registre de modèles

Le script d'entraînement journalise les métriques avec [MLflow Tracking](https://mlflow.org/docs/latest/tracking.html) (`autolog`), ainsi que le jeu de données d'entraînement. Il enveloppe ensuite le modèle Keras dans un modèle `pyfunc` personnalisé (`WrappedModel`) qui embarque le prétraitement des images, le publie sur le registre de modèles sous le nom `dev.ml.landscape-classifier`, et donne l'alias `champion` à la nouvelle version.

In [ ]:
!cat src/landscape_classifier/train.py

Ses paramètres sont gérés par DVC, dans `params.yaml`&nbsp;:

In [ ]:
!cat params.yaml

In [ ]:
!landscape-classifier

## Utilisation du registre de modèles MLflow

### Création d'un client MLflow

L'adresse de tracking est bien récupérée depuis la variable d'environnement définie en haut du notebook&nbsp;:

In [ ]:
import mlflow

MODEL_NAME = "dev.ml.landscape-classifier"

client = mlflow.MlflowClient()
client.tracking_uri

### Recherche de modèles

Tous les modèles enregistrés, avec pour chacun sa version la plus récente&nbsp;:

In [ ]:
def latest_version(name: str) -> str:
  versions = client.search_model_versions(f"name = '{name}'")
  return str(max(int(v.version) for v in versions))


for registered_model in client.search_registered_models():
  print(registered_model.name, latest_version(registered_model.name))

### Récupération d'un modèle depuis le registre de modèles

Un modèle MLflow peut avoir plusieurs *flavors*, autant de manières de le charger. Le nôtre a été publié comme un modèle `pyfunc` personnalisé&nbsp;: sa seule flavor est `python_function`, qui donne tout le pipeline (prétraitement des images, prédiction, mise en forme du résultat). Une flavor Keras n'aurait donné que le réseau, sans le prétraitement.

In [ ]:
from pathlib import Path

version = latest_version(MODEL_NAME)
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{version}")
print(list(model.metadata.flavors))
model.predict([Path("forest1.jpg").read_bytes(), Path("forest2.jpg").read_bytes()])

### Ajout d'alias et d'étiquettes

Les alias et les étiquettes (tags) permettent d'organiser les modèles dans le registre de modèles MLflow. Les alias ciblent forcément une version en particulier alors que les étiquettes peuvent aussi cibler toutes les versions d'un modèle.

Le script d'entraînement a déjà donné l'alias `champion` à la dernière version. On ajoute un tag `domain` à la valeur `cv` au modèle et un tag `size` à la valeur `small` à sa dernière version&nbsp;:

In [ ]:
client.set_registered_model_tag(MODEL_NAME, "domain", "cv")
client.set_model_version_tag(MODEL_NAME, version, "size", "small")
client.get_model_version_by_alias(MODEL_NAME, "champion")

On peut alors récupérer le modèle par son alias plutôt que par sa version exacte&nbsp;:

In [ ]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}@champion")
model.predict([Path("forest1.jpg").read_bytes()])

### Changement de phase d'un modèle

Pour promouvoir un modèle dans la chaîne de mise en production, la méthode recommandée est de copier la version voulue vers un nouveau nom de modèle, ici celui de l'environnement de préproduction&nbsp;:

In [ ]:
client.copy_model_version(
  src_model_uri=f"models:/{MODEL_NAME}@champion",
  dst_name="staging.ml.landscape-classifier",
)

## Création d'une API avec FastAPI

Le prétraitement est la fonction `process_image`, utilisée aussi bien à l'entraînement qu'à l'inférence (dans `WrappedModel`)&nbsp;: c'est le point auquel il faut faire extrêmement attention, un prétraitement différent entre entraînement et inférence dégrade les prédictions sans la moindre erreur.

In [ ]:
!cat src/landscape_classifier/data.py

L'API charge le modèle une seule fois, au démarrage, depuis le registre (par son alias), décrit son type de retour avec [Pydantic](https://docs.pydantic.dev/latest/) et accepte des images [envoyées en POST](https://fastapi.tiangolo.com/tutorial/request-files/)&nbsp;:

In [ ]:
!cat src/landscape_classifier/api.py

### Lancement de l'API

On lance le serveur en arrière-plan, puis on attend qu'il réponde (le chargement du modèle peut prendre un moment)&nbsp;:

In [ ]:
!nohup uvicorn landscape_classifier.api:app --port 8000 > api.log 2>&1 &

In [ ]:
import time

import requests

for _ in range(300):
  try:
    requests.get("http://127.0.0.1:8000/docs", timeout=1)
    break
  except requests.ConnectionError:
    time.sleep(1)
print(Path("api.log").read_text())

### Appel de l'API

In [ ]:
with open("forest1.jpg", "rb") as forest1, open("forest2.jpg", "rb") as forest2:
  response = requests.post(
    "http://127.0.0.1:8000/",
    files=[("images", forest1), ("images", forest2)],
  )
response.raise_for_status()
response.json()

## Création d'un Dockerfile

Docker n'est pas disponible dans Colab&nbsp;: voici le Dockerfile de la solution. Une première étape exporte les dépendances avec `uv`, la seconde les installe sur l'image de base Python 3.12 avec le projet, puis lance le serveur FastAPI.

In [ ]:
!cat Dockerfile

Sur une machine avec Docker, on construit puis lance l'image en lui passant les identifiants MLflow en variables d'environnement&nbsp;:

    docker build -t shuuchuu/landscape-classifier .
    docker run --rm -it -e MLFLOW_TRACKING_URI -e MLFLOW_TRACKING_USERNAME \
      -e MLFLOW_TRACKING_PASSWORD -p 8000:80 shuuchuu/landscape-classifier

La documentation de l'API, qui sert aussi de client de test, est alors à l'adresse <http://localhost:8000/docs>.

## Amélioration de l'API

Plusieurs choix de la solution rendent l'API adaptée à la mise en production par image Docker&nbsp;:

- le modèle est chargé une seule fois, au démarrage, et non à chaque requête&nbsp;;
- les identifiants MLflow sont passés au lancement du conteneur, jamais copiés dans l'image&nbsp;;
- l'API répond 503 si le modèle n'a pas pu être chargé.

On pourrait aller plus loin en rendant le nom et l'alias du modèle configurables par variables d'environnement, pour servir la même image en développement et en préproduction.